# [GĐ2] 04b – Dự đoán test & xuất file bàn giao cho TV4

Nạp `results/bilstm_best_model.pt` và `results/bilstm_vocab.pkl` (tạo bởi 04a), dự đoán tập test, đánh giá, rồi xuất **`results/bilstm_predictions.csv`**.

Notebook này **không cần train lại**, có thể chạy độc lập.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix

from src.bilstm_utils import (
    Vocab, make_loader, predict_proba, compute_metrics, load_checkpoint,
)

DATA_DIR    = "../data"
RESULTS_DIR = "../results"
MODEL_PATH  = os.path.join(RESULTS_DIR, "bilstm_best_model.pt")
VOCAB_PATH  = os.path.join(RESULTS_DIR, "bilstm_vocab.pkl")
PRED_PATH   = os.path.join(RESULTS_DIR, "bilstm_predictions.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Nạp vocab, mô hình và ngưỡng

In [2]:
vocab = Vocab.load(VOCAB_PATH)
model, ckpt = load_checkpoint(MODEL_PATH, device)
cfg = ckpt["cfg"]
THRESHOLD = ckpt.get("threshold", 0.5)

print("Vocab size     :", len(vocab))
print("Best epoch     :", ckpt.get("epoch"))
print("Ngưỡng sử dụng :", THRESHOLD, "(chọn trên val ở notebook 04a)")
print("Model config   :", ckpt["model_config"])

Vocab size     : 3182
Best epoch     : 4
Ngưỡng sử dụng : 0.17 (chọn trên val ở notebook 04a)
Model config   : {'vocab_size': 3182, 'embed_dim': 100, 'hidden_dim': 128, 'num_layers': 1, 'dropout': 0.5}


## 2. Dự đoán trên tập test

In [3]:
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
test_df[cfg["TEXT_COL"]] = test_df[cfg["TEXT_COL"]].fillna("").astype(str)
has_label = cfg["LABEL_COL"] in test_df.columns

test_loader = make_loader(
    test_df, vocab, text_col=cfg["TEXT_COL"],
    label_col=cfg["LABEL_COL"] if has_label else None,
    max_len=cfg["MAX_LEN"], batch_size=128, shuffle=False,
)
probs, y_true, _ = predict_proba(model, test_loader, device)
y_pred = (probs >= THRESHOLD).astype(int)
print(f"Đã dự đoán {len(probs)} mẫu | số mẫu dự đoán toxic: {y_pred.sum()}")

Đã dự đoán 2000 mẫu | số mẫu dự đoán toxic: 67


## 3. Đánh giá trên test (nhãn `human_consensus`)

In [4]:
if has_label:
    rows = {
        "ngưỡng 0.5": compute_metrics(y_true, probs, 0.5),
        f"ngưỡng val ({THRESHOLD})": compute_metrics(y_true, probs, THRESHOLD),
    }
    display(pd.DataFrame(rows).round(4))

    print(classification_report(y_true, y_pred, target_names=["non-toxic", "toxic"], digits=4))
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    display(pd.DataFrame(cm, index=["thật: non-toxic", "thật: toxic"],
                         columns=["đoán: non-toxic", "đoán: toxic"]))

,ngưỡng 0.5,ngưỡng val (0.17)
threshold,0.5000,0.1700
f1_toxic,0.1132,0.2617
precision_toxic,0.2308,0.2090
recall_toxic,0.0750,0.3500
f1_macro,0.5506,0.6207
pr_auc,0.1878,0.1878
roc_auc,0.9201,0.9201
tp,3.0000,14.0000
fp,10.0000,53.0000
fn,37.0000,26.0000


              precision    recall  f1-score   support

   non-toxic     0.9865    0.9730    0.9797      1960
       toxic     0.2090    0.3500    0.2617        40

    accuracy                         0.9605      2000
   macro avg     0.5978    0.6615    0.6207      2000
weighted avg     0.9710    0.9605    0.9653      2000



,đoán: non-toxic,đoán: toxic
thật: non-toxic,1907,53
thật: toxic,26,14


> Báo cáo chính nên dùng **F1 / Precision / Recall lớp toxic**, **F1-macro** và **PR-AUC**. Accuracy luôn khoảng 98% kể cả khi mô hình đoán toàn "non-toxic", nên không có ý nghĩa ở đây.

## 4. Xuất file bàn giao cho TV4

In [5]:
keep_cols = [c for c in ["review_id", "product_id", "nganh_hang", "so_sao",
                         "noi_dung_goc", cfg["TEXT_COL"]] if c in test_df.columns]
pred_df = test_df[keep_cols].copy()
if has_label:
    pred_df["label_true"] = y_true
pred_df["prob_toxic"] = np.round(probs, 6)
pred_df["label_pred"] = y_pred
pred_df["threshold"]  = THRESHOLD
pred_df["model"]      = "bilstm"

pred_df.to_csv(PRED_PATH, index=False, encoding="utf-8-sig")   # utf-8-sig: Excel đọc đúng tiếng Việt
print("Đã lưu ->", PRED_PATH, "|", pred_df.shape)
pred_df.head()

Đã lưu -> ../results\bilstm_predictions.csv | (2000, 11)


,review_id,product_id,nganh_hang,so_sao,noi_dung_goc,noi_dung_tien_xu_ly,label_true,prob_toxic,label_pred,threshold,model
0,58896069,325003,may-loc-nuoc,2,Chính sách bảo hành và hỗ trợ đồng bào của hãn...,chính_sách bảo_hành hỗ_trợ đồng_bào hãng đối_v...,0,0.466838,1,0.17,bilstm
1,33055842,88889,bep-ga,4,Mới đầu đun lúc nhấc nồi ra thì kiềng dính vào...,mới_đầu đun nhấc nồi kiềng dính đay nồi luôn k...,0,0.000235,0,0.17,bilstm
2,54635288,88053,tai-nghe,4,"Hình thức đẹp, âm thanh tốt.",hình_thức đẹp âm_thanh tốt,0,0.000043,0,0.17,bilstm
3,59468863,357878,may-loc-nuoc,5,"Máy dùng tốt, sẽ giới thiệu thêm cho người thân",máy dùng tốt giới_thiệu thêm người_thân,0,0.000158,0,0.17,bilstm
4,57265366,272138,quat-dieu-hoa,1,"Tôi mua giá hơn 3 triệu, sử dụng được 4 tháng ...",mua giá hơn 3 triệu sử_dụng được 4 tháng đèn c...,1,0.003550,0,0.17,bilstm


**Các cột trong file:** `review_id` (khoá để TV4 ghép với kết quả mô hình khác), `label_true`, `prob_toxic` (xác suất, để TV4 tự đổi ngưỡng hoặc vẽ PR curve), `label_pred` (nhãn theo ngưỡng đã chọn trên val), `threshold`, `model`.

## 5. Phân tích lỗi (để viết báo cáo)

In [6]:
if has_label:
    pd.set_option("display.max_colwidth", 200)
    fn = pred_df[(pred_df.label_true == 1) & (pred_df.label_pred == 0)].sort_values("prob_toxic")
    fp = pred_df[(pred_df.label_true == 0) & (pred_df.label_pred == 1)].sort_values("prob_toxic", ascending=False)
    print(f"False Negative (toxic bị bỏ sót): {len(fn)}")
    display(fn[["so_sao", "noi_dung_goc", "prob_toxic"]].head(10))
    print(f"False Positive (báo động nhầm): {len(fp)}")
    display(fp[["so_sao", "noi_dung_goc", "prob_toxic"]].head(10))

False Negative (toxic bị bỏ sót): 26


,so_sao,noi_dung_goc,prob_toxic
285,1,xài như hạch. được vài tháng thì bây giờ để không cũng tự tắt nguồn. phím thì bị cấn. nhấn mạnh 34 lần mới bấm được. chán,0.002305
1267,2,"đã mua rất nhiều đồ của đmx và thực sự thấy chán.bếp tự động tắt , bật. nồi chảo mua kèm theo thì cháy , nung. mất an toàn. chế độ bảo hành thì hời hợt. quá thất vọng",0.002762
4,1,"Tôi mua giá hơn 3 triệu, sử dụng được 4 tháng đèn cảm ứng sáng loạn xạ, bật hay tắt gì vẫn sáng, nút chế độ lạnh thì không sáng. Gọi bảo hành thì bảo phải mang máy về hãng sửa tầm 15 ngày, vậy 15 ...",0.003550
1253,2,"Bếp lữa đỏ xài quá hao ga, 2 tháng 1 bình ga, nấu ăn không thường. Quá thất vọng với sản phẩm này",0.003906
1437,1,"Rất tệ, giá mắc hơn junger, không nấu liu riu được, chỉnh mức nhiệt nhỏ nhất 60độ vẫn sôi trào.",0.005087
802,1,Dùng con điều hoà đểu 8 năm k hỏng.Dùng con này đc đúng 2 năm thì hỏng.Đắt nhất trong các dòng mà độ bền lởm vcđ,0.009448
1577,1,Hẹn giao hàng 15h. Hơn 16h không một ai liên hệ. Hơn 17h mới giao hàng tới nơi. Làm ảnh hưởng rất nhiều tới thời gian và công việc. Trải nghiệm quá tệ.,0.016044
1857,1,"Cùng phân khúc 67tr mà sản xuất cái điều hòa như hạchĐiều hòa gì mà chức năng cơ bản như chế độ auto, hút ẩm, cải thiện chất lượng không khí, Hay đơn giản cơ bản nhất là cái đảo chiều gió trái phả...",0.025573
1234,3,Tại sao máy t mua online trên áp thang 2-2022 mà lại xuất kho tháng 2-2021 Phải chăng máy cũ đã qua xử dụng. Đứt dây điện mà đợi gần 1 tuần hãng ko liên lạc lại để xử lý.phục vụ quá tệ hãng panasonic,0.031481
1080,1,Thực tế chỉ được 250MbpsKhông bắt được sóng 5GBịp,0.031843


False Positive (báo động nhầm): 53


,so_sao,noi_dung_goc,prob_toxic
891,1,"Lần 3 Vẫn không thấy Kỹ thuật của Hảng Sharp liên hệ bảo hành, ĐMX và Sharp xem lại, nếu đơn thuần mục đích chạy doanh số mà không quan tâm đến hậu mãi thì chắc không phù hợp thị trường hiện nay. ...",0.995606
1967,1,"Mới mua về sạc đầy để qua đêm hết pin.Mang ra kêu để gởi bảo hành, không chấp nhận đổi mới, ko chịu hoàn tiền khi khách ko muốn mua. Kinh doanh kiểu này chắc khác nào lừa khách hàng. Mình mua rất ...",0.988305
420,1,Mọi người mua bắt điện máy xanh cam kết tới nhà bảo hành nha. khi bảo hành bắt người mua phải tới siêu thị giao và lấy về. Lúc bán thì ko nói.,0.974173
559,4,tôi thây bêp tôt hê thông đanh lưa nhay hơn bêp đăt đưng khinh bêp gia gie nka rât ưng y,0.968604
1399,1,Mua thêm gói bảo hành 1 đổi 1. Xài dc 4 tháng nồi cơm nấu bị khét. Ở trên thì k chín. Đem lại chổ mua thì nói phải gửi lên hãng xác nhận. Đưa hình chụp cơm khét thì k chịu. Nhân viên thì nói nếu k...,0.951683
1637,1,Bán điều hòa cục nóng kêu như máy sát. Gọi kêu xuống bảo hành chưa xuống mà nhân viên báo với tổng đài đã xuống và đã sử lý. Sản phẩm tệ hại. Thái độ làm việc tồi tàn,0.924473
1924,1,"Mọi người nên né, không nên mua hàng của, ĐMX",0.839104
1433,1,"bật lửa mỏi tay mới lên,ga thì tốn kinh khủng,đmx rồi đến hãng bảo hành tới lui vẫn vậy,tốn tiền mua đồ chính hãng",0.668749
1078,1,"Quạt bị kêu cạch cạch khi chạy, mình đang bố trí thời gian qua điện máy xanh để đổi",0.584951
1337,1,Mua cái trúng cái loa bass cứ ù ù ta nói nó tức Mọi người đừng mua loa này phí tiền,0.509420


## 6. (Tuỳ chọn) Dự đoán một câu mới

Câu đầu vào phải được tiền xử lý **giống hệt** cột `noi_dung_tien_xu_ly` (chuẩn hoá teencode, tách từ, bỏ stopword…), nếu không kết quả sẽ sai lệch. Hãy dùng lại hàm tiền xử lý của nhóm ở GĐ1.

In [7]:
def predict_texts(texts_preprocessed, threshold=THRESHOLD):
    df = pd.DataFrame({cfg["TEXT_COL"]: texts_preprocessed})
    loader = make_loader(df, vocab, text_col=cfg["TEXT_COL"], label_col=None,
                         max_len=cfg["MAX_LEN"], batch_size=64)
    p, _, _ = predict_proba(model, loader, device)
    return pd.DataFrame({"text": texts_preprocessed, "prob_toxic": p.round(4),
                         "label_pred": (p >= threshold).astype(int)})

# Ví dụ (text đã tách từ):
predict_texts(["sản_phẩm tốt giao hàng nhanh", "điện_máy_xanh làm_ăn lừa_đảo khách_hàng"])

,text,prob_toxic,label_pred
0,sản_phẩm tốt giao hàng nhanh,0.0005,0
1,điện_máy_xanh làm_ăn lừa_đảo khách_hàng,0.9892,1
